In [ ]:
from utils import *
import os
#os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
#os.environ["CUDA_VISIBLE_DEVICES"]="6"

In [ ]:
root_directory = "/nfs/datz/olmo_models/neuron_outputs2"

file_paths = traverse_directory(root_directory)
#main_files = [path for path in file_paths if ("subgraph" not in path and "test" not in path and "all_country_capital" in path)]

all_files = [file for file in file_paths if "country_capital_city" in file]

In [ ]:
all_files

In [ ]:
all_files = all_files[:10]
all_files.append("/nfs/datz/olmo_models/neuron_outputs/main/country_capital_city.pkl")

In [ ]:
# root_directory = "/nfs/datz/olmo_models/neuron_outputs"

# file_paths = traverse_directory(root_directory)
# #main_files = [path for path in file_paths if ("subgraph" not in path and "test" not in path and "all_country_capital" in path)]

# all_files = [file for file in file_paths if "test" not in file]

models_data = read_all_files(all_files)

for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")
        
sorted_models = sorted(
        models_data.keys(), 
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )

In [ ]:
model_name = "allenai/OLMo-7B-0424-hf"

model = load_models(model_name, device='cuda:7', checkpoint="main")

In [ ]:
models_data['main']['country_capital_city'][2]

In [ ]:
heatmaps_dict = {}

theta = 0.01

for model_name in sorted_models:
    relations_data = models_data[model_name]

    print(model_name)
    g_total = 0
    general_heatmap = np.zeros((32, 11008), dtype=float)
    e_total = 0
    entity_heatmap = np.zeros((32, 11008), dtype=float)

    relation_answer_heatmaps = {}
    relation_answer_with_specific = {}

    for relation_name, entries in relations_data.items():

        relation_answer_heatmap = np.zeros((32, 11008), dtype=float)
        relation_answer_total = 0
        answer_specific_heatmaps = {}

        for idx, entry in enumerate(entries):

            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()

            # Update general heatmap
            #entry['neuron_contributions']['vals'][:17].shape
            #general_heatmap += np.mean(entry['neuron_contributions']['vals'][:entry['answer_token_span'][0]], axis=0) #added [:entry['answer_token_span'][0]] because I dont need the contributions from the answer token itself
            general_heatmap += torch.mean(entry['neuron_contributions']['vals'], dim=0).cpu().numpy()
            g_total += 1

            # Update entity heatmap for subject tokens
            one_data_map = np.zeros((32, 11008), dtype=float)
            for t in entry["subj_token_span"]:
                # if t == 0:
                #     e_total -= 1
                #     continue
                one_data_map += entry['neuron_contributions']['vals'][t].cpu().numpy()
            entity_heatmap += one_data_map
            e_total += len(entry["subj_token_span"])

            # Update entity heatmap and relation answer heatmap for answer tokens
            one_data_map = np.zeros((32, 11008), dtype=float)
            for t in entry["answer_token_span"]:
                one_data_map += entry['neuron_contributions']['vals'][t - 1].cpu().numpy()
            entity_heatmap += one_data_map
            relation_answer_heatmap += one_data_map

            e_total += len(entry["answer_token_span"])
            relation_answer_total += len(entry["answer_token_span"])

            # Store answer-specific heatmap
            answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
            answer_specific_heatmaps[idx] = answer_specific_heatmap  > theta

        relation_answer_heatmap /= relation_answer_total
        relation_answer_heatmaps[f"{relation_name}"] = relation_answer_heatmap > theta
        relation_answer_with_specific[f"{relation_name}"] = answer_specific_heatmaps

    # Normalize heatmaps
    general_heatmap /= g_total
    entity_heatmap /= e_total

    # Store heatmaps in the dictionary
    heatmaps_dict[model_name] = {
        'general_heatmap': general_heatmap > theta,
        'entity_heatmap': entity_heatmap > theta,
        'relation_answer_heatmaps': relation_answer_heatmaps,
        'relation_answer_with_specific': relation_answer_with_specific
    }

In [ ]:
def calculate_proper_heads(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.

    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and 
                              the value is a dictionary with heatmaps.

    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heads = heatmaps['relation_answer_heatmaps']
        answer_specific_heads = heatmaps['relation_answer_with_specific']

        # Calculate proper heads
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Initialize dictionaries to store results for relations and specific answers
        proper_relation_answer_heads = {}
        proper_answer_specific_heads = {}

        # Calculate proper relation answer heads per relation
        for relation, relation_heads in relation_answer_heatmaps.items():
            proper_relation_answer_heads[relation] = np.logical_and(
                relation_heads,
                np.logical_not(np.logical_or(entity_heads, general_heads))
            )

        # Calculate proper answer specific heads per relation and specific answer
        for relation, specific_answers in relation_answer_with_specific.items():
            proper_answer_specific_heads[relation] = {}
            for answer_id, specific_heads in specific_answers.items():
                proper_answer_specific_heads[relation][answer_id] = np.logical_and(
                    specific_heads,
                    np.logical_not(
                        np.logical_or(
                            relation_answer_heatmaps[relation],
                            np.logical_or(entity_heads, general_heads)
                        )
                    )
                )

        # Store results in the output dictionary
        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

In [ ]:
def calculate_consistency(heatmaps_dict, relation_name, fact_idx):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store IoU results for this model
        model_ious = {}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "relation_answer_heatmaps":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
            elif heatmap_type == "entity_heatmap":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            elif heatmap_type == "general_heatmap":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            else:
                continue

            # Calculate IoU
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)

            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)

            # Avoid division by zero
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

            # Store IoU result and statistics
            model_ious[heatmap_type] = {
                'iou': iou,
                'intersection': area_of_intersection,
                'union': area_of_union
            }

        # Add this model's IoU results to the main dictionary
        iou_results[model_name] = model_ious

    return iou_results

In [ ]:
# def calculate_consistency(heatmaps_dict, relation_name, fact_idx):
#     overlap_results = {}
    
#     # Retrieve the 'main' model's heatmaps
#     main_heatmaps = heatmaps_dict['main']

#     # Iterate through each model (excluding 'main') and each heatmap type
#     for model_name, model_data in heatmaps_dict.items():
#         if model_name == 'main':
#             continue

#         # Store overlaps for this model
#         model_overlaps = {}
        
#         for heatmap_type in main_heatmaps.keys():
#             if heatmap_type == "relation_answer_heatmaps":
#                 # Index by relation_name for relation_answer_heatmaps
#                 main_heatmap = (main_heatmaps[heatmap_type][relation_name])
#                 model_heatmap = (model_data[heatmap_type][relation_name])
#                 overlap_map = main_heatmap == model_heatmap
#                 count_map = np.logical_and(main_heatmap, model_heatmap)
#                 count =  np.sum(model_data[heatmap_type][relation_name])
#                 count_main = np.sum(main_heatmap)
#             elif heatmap_type == "relation_answer_with_specific":
#                 # Index by relation_name and fact_idx for relation_answer_with_specific
#                 main_heatmap = (main_heatmaps[heatmap_type][relation_name][fact_idx]) 
#                 model_heatmap = (model_data[heatmap_type][relation_name][fact_idx])
#                 overlap_map = main_heatmap == model_heatmap#(main_heatmap - model_heatmap) 
#                 count_map = np.logical_and(main_heatmap, model_heatmap)
#                 count =  np.sum(model_data[heatmap_type][relation_name][fact_idx])
#                 count_main = np.sum(main_heatmap)
#             elif heatmap_type == "entity_heatmap":
#                 main_heatmap = (main_heatmaps[heatmap_type])
#                 model_heatmap = (model_data[heatmap_type])
#                 overlap_map = main_heatmap == model_heatmap
#                 count_map = np.logical_and(main_heatmap, model_heatmap)
#                 count =  np.sum(model_data[heatmap_type])
#                 count_main = np.sum(main_heatmap)
#                 # overlap_map = np.logical_and(main_heatmap, model_heatmap)#(main_heatmap - model_heatmap) 
#                 #continue
#             elif heatmap_type == "general_heatmap":
#             #     # General or entity heatmaps
#                 main_heatmap = main_heatmaps[heatmap_type] 
#                 model_heatmap = model_data[heatmap_type]
#                 overlap_map = main_heatmap == model_heatmap
#                 count_map = np.logical_and(main_heatmap, model_heatmap)
#                 count =  np.sum(model_data[heatmap_type])
#                 count_main = np.sum(main_heatmap)
#             # Calculate summary statistics, e.g., proportion of overlapping positions
#             proportion_overlap = np.sum(overlap_map) / overlap_map.size
#             count_overlap = np.sum(count_map)
            
#             # Store the overlap map and proportion
#             model_overlaps[heatmap_type] = {
#                 'overlap_map': overlap_map,
#                 'consistency': proportion_overlap,
#                 'count_overlap': count_overlap,
#                 'count':count,
#                 'count_main': count_main
#             }
        
#         # Add this model's overlap results to the main dictionary
#         overlap_results[model_name] = model_overlaps
    
#     return overlap_results

In [ ]:
def plot_proportion_overlap_multiple(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_heatmap']['iou'])
        entity_proportions.append(overlap_results[model_name]['entity_heatmap']['iou'])
        answer_all_proportions.append(overlap_results[model_name]['relation_answer_heatmaps']['iou'])
        answer_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['iou'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Relation Answer Heatmap", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Answer Specific Heatmap", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

def plot_count_proportion_overlap_multiple(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_heatmap']['intersection'])
        entity_proportions.append(overlap_results[model_name]['entity_heatmap']['intersection'])
        answer_all_proportions.append(overlap_results[model_name]['relation_answer_heatmaps']['intersection'])
        answer_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['intersection'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Relation Answer Heatmap", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Answer Specific Heatmap", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Counts")
    plt.title(f"Counts of overlapping specialized Heads Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_all_count_proportion_overlap_multiple(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_heatmap']['union'])
        entity_proportions.append(overlap_results[model_name]['entity_heatmap']['union'])
        answer_all_proportions.append(overlap_results[model_name]['relation_answer_heatmaps']['union'])
        answer_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['union'])

    # model_names.append('main')
    # general_proportions.append(overlap_results[model_name]['general_heatmap']['count_main'])
    # entity_proportions.append(overlap_results[model_name]['entity_heatmap']['count_main'])
    # answer_all_proportions.append(overlap_results[model_name]['relation_answer_heatmaps']['count_main'])
    # answer_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['count_main'])


    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_proportions, label="General Heads", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heads", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Relation Answer Heads", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Answer Specific Heads", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Counts")
    plt.title(f"Counts of All Specialized Heads Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
models_data.keys()

In [ ]:
relation_name = "country_capital_city"
fact_idx = 1
sent = models_data['step10000-tokens41B'][relation_name][fact_idx]['sentence']
overlap = calculate_consistency(heatmaps_dict, relation_name, fact_idx)
plot_proportion_overlap_multiple(overlap, relation_name, sent)
plot_count_proportion_overlap_multiple(overlap, relation_name, sent)
plot_all_count_proportion_overlap_multiple(overlap, relation_name, sent)

In [ ]:
relation_name = "country_capital_city"
fact_idx = 1
sent = models_data['step10000-tokens41B'][relation_name][fact_idx]['sentence']
overlap = calculate_consistency(heatmaps_dict, relation_name, fact_idx)
plot_proportion_overlap_multiple(overlap, relation_name, sent)
plot_count_proportion_overlap_multiple(overlap, relation_name, sent)
plot_all_count_proportion_overlap_multiple(overlap, relation_name, sent)

In [ ]:
def calculate_proper_heads(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.

    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and 
                              the value is a dictionary with heatmaps.

    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heads = heatmaps['relation_answer_heatmaps']
        answer_specific_heads = heatmaps['relation_answer_with_specific']

        # Calculate proper heads
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Initialize dictionaries to store results for relations and specific answers
        proper_relation_answer_heads = {}
        proper_answer_specific_heads = {}

        # Calculate proper relation answer heads per relation
        for relation, relation_heads in relation_answer_heatmaps.items():
            proper_relation_answer_heads[relation] = np.logical_and(
                relation_heads,
                np.logical_not(np.logical_or(entity_heads, general_heads))
            )

        # Calculate proper answer specific heads per relation and specific answer
        for relation, specific_answers in relation_answer_with_specific.items():
            proper_answer_specific_heads[relation] = {}
            for answer_id, specific_heads in specific_answers.items():
                proper_answer_specific_heads[relation][answer_id] = np.logical_and(
                    specific_heads,
                    np.logical_not(
                        np.logical_or(
                            relation_answer_heatmaps[relation],
                            np.logical_or(entity_heads, general_heads)
                        )
                    )
                )

        # Store results in the output dictionary
        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

In [ ]:
def calculate_consistency_proper_heads(heatmaps_dict, relation_name, fact_idx):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store IoU results for this model
        model_ious = {}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
            elif heatmap_type == "proper_answer_specific_heads":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
            elif heatmap_type == "proper_entity_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            elif heatmap_type == "proper_general_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            else:
                continue

            # Calculate IoU
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)

            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)

            # Avoid division by zero
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

            # Store IoU result and statistics
            model_ious[heatmap_type] = {
                'iou': iou,
                'intersection': area_of_intersection,
                'union': area_of_union
            }

        # Add this model's IoU results to the main dictionary
        iou_results[model_name] = model_ious

    return iou_results


In [ ]:
def plot_proper_heads_iou_multiple(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_ious, label="Proper General Heads IoU", marker='o', linestyle='-')
    plt.plot(model_names, entity_ious, label="Proper Entity Heads IoU", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_ious, label="Proper Relation Answer Heads IoU", marker='x', linestyle='-.')
    plt.plot(model_names, answer_ious, label="Proper Answer Specific Heads IoU", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"Proper Heads IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_proper_heads_counts_intersection(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_intersections = []
    entity_intersections = []
    answer_all_intersections = []
    answer_intersections = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_intersections.append(iou_results[model_name]['proper_general_heads']['intersection'])
        entity_intersections.append(iou_results[model_name]['proper_entity_heads']['intersection'])
        answer_all_intersections.append(iou_results[model_name]['proper_relation_answer_heads']['intersection'])
        answer_intersections.append(iou_results[model_name]['proper_answer_specific_heads']['intersection'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_intersections, label="Proper General Heads Intersections", marker='o', linestyle='-')
    plt.plot(model_names, entity_intersections, label="Proper Entity Heads Intersections", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_intersections, label="Proper Relation Answer Heads Intersections", marker='x', linestyle='-.')
    plt.plot(model_names, answer_intersections, label="Proper Answer Specific Heads Intersections", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Intersection Counts")
    plt.title(f"Proper Heads Intersections Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_proper_heads_counts_union(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_unions = []
    entity_unions = []
    answer_all_unions = []
    answer_unions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_unions.append(iou_results[model_name]['proper_general_heads']['union'])
        entity_unions.append(iou_results[model_name]['proper_entity_heads']['union'])
        answer_all_unions.append(iou_results[model_name]['proper_relation_answer_heads']['union'])
        answer_unions.append(iou_results[model_name]['proper_answer_specific_heads']['union'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_unions, label="Proper General Heads Unions", marker='o', linestyle='-')
    plt.plot(model_names, entity_unions, label="Proper Entity Heads Unions", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_unions, label="Proper Relation Answer Heads Unions", marker='x', linestyle='-.')
    plt.plot(model_names, answer_unions, label="Proper Answer Specific Heads Unions", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Union Counts")
    plt.title(f"Proper Heads Unions Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_proportion_overlap_multiple_proper_heads(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['proper_general_heads']['consistency'])
        entity_proportions.append(overlap_results[model_name]['proper_entity_heads']['consistency'])
        answer_all_proportions.append(overlap_results[model_name]['proper_relation_answer_heads']['consistency'])
        answer_proportions.append(overlap_results[model_name]['proper_answer_specific_heads']['consistency'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_proportions, label="Proper General Heads", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Proper Entity Heads", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Proper Relation Answer Heads", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Proper Answer Specific Heads", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Consistency")
    plt.title(f"Consistency Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

def plot_count_proportion_overlap_multiple_proper_heads(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['proper_general_heads']['count_overlap'])
        entity_proportions.append(overlap_results[model_name]['proper_entity_heads']['count_overlap'])
        answer_all_proportions.append(overlap_results[model_name]['proper_relation_answer_heads']['count_overlap'])
        answer_proportions.append(overlap_results[model_name]['proper_answer_specific_heads']['count_overlap'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_proportions, label="Proper General Heads", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Proper Entity Heads", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Proper Relation Answer Heads", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Proper Answer Specific Heads", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Counts")
    plt.title(f"Counts of overlapping specialized Heads Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_all_count_proportion_overlap_multiple_proper_heads(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['proper_general_heads']['count'])
        entity_proportions.append(overlap_results[model_name]['proper_entity_heads']['count'])
        answer_all_proportions.append(overlap_results[model_name]['proper_relation_answer_heads']['count'])
        answer_proportions.append(overlap_results[model_name]['proper_answer_specific_heads']['count'])

    model_names.append('main')
    general_proportions.append(overlap_results[model_name]['proper_general_heads']['count_main'])
    entity_proportions.append(overlap_results[model_name]['proper_entity_heads']['count_main'])
    answer_all_proportions.append(overlap_results[model_name]['proper_relation_answer_heads']['count_main'])
    answer_proportions.append(overlap_results[model_name]['proper_answer_specific_heads']['count_main'])


    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_proportions, label="Proper General Heads", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Proper Entity Heads", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Proper Relation Answer Heads", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Proper Answer Specific Heads", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Counts")
    plt.title(f"Counts of all Specialized Heads Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
relation_name = "country_capital_city"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
proper_heads = calculate_proper_heads(heatmaps_dict)
overlap = calculate_consistency_proper_heads(proper_heads, relation_name, fact_idx)
plot_proper_heads_iou_multiple(overlap, relation_name, sent)
plot_proper_heads_counts_intersection(overlap, relation_name, sent)
plot_proper_heads_counts_union(overlap, relation_name, sent)

In [ ]:
relation_name = "country_capital_city"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
proper_heads = calculate_proper_heads(heatmaps_dict)
overlap = calculate_consistency_proper_heads(proper_heads, relation_name, fact_idx)
plot_proper_heads_iou_multiple(overlap, relation_name, sent)
plot_proper_heads_counts_intersection(overlap, relation_name, sent)
plot_proper_heads_counts_union(overlap, relation_name, sent)

In [ ]:
def calculate_proper_heads(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.

    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and 
                              the value is a dictionary with heatmaps.

    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heads = heatmaps['relation_answer_heatmaps']
        answer_specific_heads = heatmaps['relation_answer_with_specific']

        # Calculate proper heads
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Initialize dictionaries to store results for relations and specific answers
        proper_relation_answer_heads = {}
        proper_answer_specific_heads = {}

        # Calculate proper relation answer heads per relation
        for relation, relation_heads in relation_answer_heatmaps.items():
            proper_relation_answer_heads[relation] = np.logical_and(
                relation_heads,
                np.logical_not(np.logical_or(entity_heads, general_heads))
            )

        # Calculate proper answer specific heads per relation and specific answer
        for relation, specific_answers in relation_answer_with_specific.items():
            proper_answer_specific_heads[relation] = {}
            for answer_id, specific_heads in specific_answers.items():
                proper_answer_specific_heads[relation][answer_id] = np.logical_and(
                    specific_heads,
                    np.logical_not(
                        np.logical_or(
                            relation_answer_heatmaps[relation],
                            np.logical_or(entity_heads, general_heads)
                        )
                    )
                )

        # Store results in the output dictionary
        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

def calculate_consistency_proper_heads_all_facts(heatmaps_dict, relation_name):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU, intersection, and union results for this model
        model_metrics = {heatmap_type: {'iou': [], 'head_count': [], 'intersection': [], 'union': []}
                         for heatmap_type in main_heatmaps.keys()}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Iterate over all facts (assume array structure)
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            elif heatmap_type == "proper_answer_specific_heads":
                # Iterate over all facts and answers
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                    #for answer_id, main_specific_heatmap in enumerate(
                    #    main_heatmaps[heatmap_type][relation_name][fact_idx]
                    #~):
                main_specific_heatmap = np.logical_or.reduce(list(main_heatmaps[heatmap_type][relation_name].values())) #main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_specific_heatmap = np.logical_or.reduce(list(model_data[heatmap_type][relation_name].values())) #model_data[heatmap_type][relation_name][fact_idx]#[answer_id]

                # Calculate metrics
                intersection = np.logical_and(main_specific_heatmap, model_specific_heatmap)
                union = np.logical_or(main_specific_heatmap, model_specific_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_specific_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            else:
                # Handle proper_entity_heads and proper_general_heads
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

        # Aggregate results for each heatmap type
        aggregated_metrics = {ht: {
            'iou': np.mean(metrics['iou']),
            'head_count': np.sum(metrics['head_count']),
            'intersection': np.sum(metrics['intersection']),
            'union': np.sum(metrics['union'])
        } for ht, metrics in model_metrics.items()}

        iou_results[model_name] = aggregated_metrics

    return iou_results

def plot_aggregated_proper_heads_iou(iou_results, relation_name):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []

    # Populate lists for each heatmap type
    for model_name in model_names[:-1]:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

    for model_name in model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

    # Create directories if not exist
    #iou_dir = os.path.join(output_dir, "iou")
    #count_dir = os.path.join(output_dir, "count")
    #os.makedirs(iou_dir, exist_ok=True)
    #os.makedirs(count_dir, exist_ok=True)

    # Plot and save IoU
    plt.figure(figsize=(20, 6))
    plt.plot(model_names[:-1], general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
    plt.plot(model_names[:-1], entity_ious, label="Proper Entity Heads IoU (Aggregated)", marker='s', linestyle='--')
    plt.plot(model_names[:-1], answer_all_ious, label="Proper Relation Answer Heads IoU (Aggregated)", marker='x', linestyle='-.')
    plt.plot(model_names[:-1], answer_ious, label="Proper Answer Specific Heads IoU (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names[:-1])), labels=model_names[:-1], rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated IoU")
    plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    #plt.savefig(os.path.join(iou_dir, f"{relation_name}_heads_iou.png"))
    plt.close()

    # Plot and save Count
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
    plt.plot(model_names, entity_head_count, label="Proper Entity Heads Count (Aggregated)", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_head_count, label="Proper Relation Answer Heads Count (Aggregated)", marker='x', linestyle='-.')
    plt.plot(model_names, answer_head_counts, label="Proper Answer Specific Heads Count (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated Count")
    plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    #plt.savefig(os.path.join(count_dir, f"{relation_name}_heads_count.png"))
    plt.show()
    plt.close()


# def plot_proper_heads_counts_intersection(iou_results, relation_name):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )

#     general_intersections = []
#     entity_intersections = []
#     answer_all_intersections = []
#     answer_intersections = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_intersections.append(iou_results[model_name]['proper_general_heads']['intersection'])
#         entity_intersections.append(iou_results[model_name]['proper_entity_heads']['intersection'])
#         answer_all_intersections.append(iou_results[model_name]['proper_relation_answer_heads']['intersection'])
#         answer_intersections.append(iou_results[model_name]['proper_answer_specific_heads']['intersection'])

#     # Plotting Intersections
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_intersections, label="Proper General Neurons Intersections", marker='o', linestyle='-')
#     plt.plot(model_names, entity_intersections, label="Proper Entity Neurons Intersections", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_intersections, label="Proper Relation Answer Neurons Intersections", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_intersections, label="Proper Answer Specific Neurons Intersections", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Intersection Counts")
#     plt.title(f"Proper Neurons Intersections Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()


# def plot_proper_heads_counts_union(iou_results, relation_name):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )

#     general_unions = []
#     entity_unions = []
#     answer_all_unions = []
#     answer_unions = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_unions.append(iou_results[model_name]['proper_general_heads']['union'])
#         entity_unions.append(iou_results[model_name]['proper_entity_heads']['union'])
#         answer_all_unions.append(iou_results[model_name]['proper_relation_answer_heads']['union'])
#         answer_unions.append(iou_results[model_name]['proper_answer_specific_heads']['union'])

#     # Plotting Unions
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_unions, label="Proper General Neurons Unions", marker='o', linestyle='-')
#     plt.plot(model_names, entity_unions, label="Proper Entity Neurons Unions", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_unions, label="Proper Relation Answer Neurons Unions", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_unions, label="Proper Answer Specific Neurons Unions", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Union Counts")
#     plt.title(f"Proper Neurons Unions Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()

In [ ]:
relation_names = models_data['main'].keys()

In [ ]:
for relation_name in relation_names:
    # Extract the sentences for all facts (optional, for reference)
    #sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

    # Step 1: Calculate proper heads
    proper_heads = calculate_proper_heads(heatmaps_dict)

    # Step 2: Calculate overlap (IoU) across all facts for the relation
    overlap = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

    # Step 3: Plot aggregated IoU across all facts for the relation
    plot_aggregated_proper_heads_iou(overlap, relation_name)

    # Step 4 (Optional): Add plots for intersections and unions if needed
    #plot_proper_heads_counts_intersection(overlap, relation_name)
    #plot_proper_heads_counts_union(overlap, relation_name)

In [ ]:
relation_name = "country_capital_city"

# Extract the sentences for all facts (optional, for reference)
#sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

# Step 1: Calculate proper heads
proper_heads = calculate_proper_heads(heatmaps_dict)

# Step 2: Calculate overlap (IoU) across all facts for the relation
overlap = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

# Step 3: Plot aggregated IoU across all facts for the relation
plot_aggregated_proper_heads_iou(overlap, relation_name)

# Step 4 (Optional): Add plots for intersections and unions if needed
plot_proper_heads_counts_intersection(overlap, relation_name)
plot_proper_heads_counts_union(overlap, relation_name)